In [ ]:
from collections import Counter
from pathlib import Path
import gc
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'experiment':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'viskom_pipeline.py').exists():
    raise RuntimeError('Open this notebook from the project root or experiment folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viskom_pipeline import (
    _class_color, _class_name, _draw_mask_annotations, _mask_overlay,
    _resize_visualization_inputs, clear_analysis_model_cache,
    load_analysis_bundle, predict_image_file,
)

DEMO_IMAGE = PROJECT_ROOT / 'My_Dataset' / 'demo_image' / 'image.png'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'full_runs'
CLASS_SCHEMA_ROOT = PROJECT_ROOT / 'Roboflow_Dataset'
OUTPUT_ROOT = (PROJECT_ROOT / 'analysis' / 'figures' / 'demo' /
               'pre_experiment_six_models')
SCORE_THRESHOLD = 0.25
MAX_LABELS_PER_PANEL = 14

MODEL_SPECS = [
    ('yolo26n_seg_pretrained', 'YOLO26n-Seg — Pretrained'),
    ('yolo26n_seg_scratch', 'YOLO26n-Seg — Scratch'),
    ('maskrcnn_r50_fpn_v2_pretrained', 'Mask R-CNN R50-FPN v2 — Pretrained'),
    ('maskrcnn_r50_fpn_v2_scratch', 'Mask R-CNN R50-FPN v2 — Scratch'),
    ('mask2former_swin_tiny_pretrained', 'Mask2Former Swin-T — Pretrained'),
    ('mask2former_swin_tiny_scratch', 'Mask2Former Swin-T — Scratch'),
]

if not DEMO_IMAGE.exists():
    raise FileNotFoundError(f'Demo image not found: {DEMO_IMAGE}')
missing_runs = [run_id for run_id, _ in MODEL_SPECS
                if not (ARTIFACT_ROOT / run_id / 'run_metadata.json').exists()]
if missing_runs:
    raise FileNotFoundError(f'Missing completed run metadata: {missing_runs}')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Image:', DEMO_IMAGE)
print('Confidence threshold:', SCORE_THRESHOLD)
print('CUDA available:', torch.cuda.is_available())
print('Output:', OUTPUT_ROOT)


## Run all six pre-experiment checkpoints

This may take several minutes. The models are intentionally loaded sequentially.


In [ ]:
prediction_panels = []
summary_rows = []
display_original = None

for model_index, (run_id, display_name) in enumerate(MODEL_SPECS, start=1):
    print(f'[{model_index}/6] Loading {display_name}...')
    bundle = load_analysis_bundle(run_id, ARTIFACT_ROOT, CLASS_SCHEMA_ROOT)
    try:
        image, prediction = predict_image_file(
            bundle, DEMO_IMAGE, score_threshold=SCORE_THRESHOLD)
        display_image, _, display_prediction = _resize_visualization_inputs(
            image, prediction=prediction, max_edge=1280)
        if display_original is None:
            display_original = display_image.permute(1, 2, 0).cpu().numpy()
        overlay = _mask_overlay(
            display_image, display_prediction['masks'],
            display_prediction['labels'])
        labels = [int(value) for value in display_prediction['labels']]
        scores = [float(value) for value in display_prediction['scores']]
        class_names = [bundle['class_names'][label] for label in labels]
        class_counts = Counter(class_names)

        individual_path = OUTPUT_ROOT / f'{model_index:02d}_{run_id}.png'
        individual_fig, individual_axis = plt.subplots(figsize=(8, 8))
        individual_axis.imshow(overlay)
        individual_axis.set_title(
            f'{display_name}\n{len(labels)} predictions | confidence ≥ {SCORE_THRESHOLD}',
            fontsize=12, fontweight='bold')
        _draw_mask_annotations(
            individual_axis, display_prediction['masks'],
            [f'{_class_name(bundle["class_names"], label)} {score:.2f}'
             for label, score in zip(labels, scores)],
            [_class_color(label) for label in labels],
            fontsize=7, max_labels=MAX_LABELS_PER_PANEL)
        individual_axis.axis('off')
        individual_fig.tight_layout()
        individual_fig.savefig(individual_path, dpi=180, bbox_inches='tight')
        plt.close(individual_fig)

        prediction_panels.append({
            'run_id': run_id, 'display_name': display_name,
            'overlay': overlay, 'masks': display_prediction['masks'],
            'labels': labels, 'scores': scores,
            'class_names': list(bundle['class_names']),
            'individual_path': individual_path,
        })
        summary_rows.append({
            'run_id': run_id, 'model': display_name,
            'initialization': bundle['run']['initialization'],
            'num_predictions': len(labels),
            'mean_confidence': float(np.mean(scores)) if scores else np.nan,
            'max_confidence': float(np.max(scores)) if scores else np.nan,
            'predicted_classes': '; '.join(
                f'{name}:{count}' for name, count in class_counts.most_common()),
            'prediction_image': str(individual_path),
        })
        print(f'    {len(labels)} predictions saved to {individual_path.name}')
    finally:
        del bundle
        gc.collect()
        clear_analysis_model_cache()

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_ROOT / 'six_model_demo_summary.csv', index=False)
display(summary)


## One original plus six predictions

The final panel is deliberately left as an explanation box. There are exactly seven image panels: one original and six model predictions.


In [ ]:
if len(prediction_panels) != 6:
    raise RuntimeError(f'Expected six prediction panels, found {len(prediction_panels)}.')

fig, axes = plt.subplots(2, 4, figsize=(22, 12))
flat_axes = axes.ravel()
flat_axes[0].imshow(display_original)
flat_axes[0].set_title('Original — no ground truth', fontsize=12, fontweight='bold')
flat_axes[0].axis('off')

for axis, panel in zip(flat_axes[1:7], prediction_panels):
    axis.imshow(panel['overlay'])
    axis.set_title(
        f'{panel["display_name"]}\n{len(panel["labels"])} predictions',
        fontsize=11, fontweight='bold')
    _draw_mask_annotations(
        axis, panel['masks'],
        [f'{_class_name(panel["class_names"], label)} {score:.2f}'
         for label, score in zip(panel['labels'], panel['scores'])],
        [_class_color(label) for label in panel['labels']],
        fontsize=6, max_labels=MAX_LABELS_PER_PANEL)
    axis.axis('off')

flat_axes[7].axis('off')
flat_axes[7].text(
    0.5, 0.60, 'QUALITATIVE DEMO', ha='center', va='center',
    fontsize=18, fontweight='bold', transform=flat_axes[7].transAxes)
flat_axes[7].text(
    0.5, 0.42,
    'Same new photo\nSame confidence threshold\nOriginal 74-class checkpoints\nNo ground truth → no mAP/F1',
    ha='center', va='center', fontsize=13, linespacing=1.5,
    transform=flat_axes[7].transAxes)

fig.suptitle(
    f'Live Inference Before Wall-Merge Experiment — confidence ≥ {SCORE_THRESHOLD}',
    fontsize=17, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.96))
comparison_path = OUTPUT_ROOT / 'original_plus_six_predictions.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Combined comparison:', comparison_path)
print('Six individual predictions:', OUTPUT_ROOT)
print('IMPORTANT: qualitative output only; do not report it as accuracy.')
